In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df=pd.read_csv('IPL.csv')

In [ ]:
df=df[df['batter']=='V Kohli']

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df = df[df["date"].dt.year <= 2023]

In [ ]:
def get_phase(over):
    if over <= 6:
        return "Powerplay"
    elif over <= 15:
        return "Middle"
    else:
        return "Death"

df["over_phase"] = df["over"].apply(get_phase)

In [ ]:
df = df.drop(df[df["runs_total"] == 7].index)
df['runs_total']=np.where(df['runs_total']>3,1,0)

In [ ]:
df['runs_total'].value_counts()

In [ ]:
dfi1 = df[df['innings'] == 1].copy()
dfi2 = df[df['innings'] == 2].copy()

In [ ]:
dfi1['runs_total'].value_counts()

In [ ]:
import numpy as np
import pandas as pd

def create_dataset(dfi1):
    dfi1 = dfi1.sort_values(
        ['match_id', 'over', 'ball']
    ).reset_index(drop=True)
    
    # Current Strike Rate (before current ball)
    dfi1['current_sr'] = (
        dfi1.groupby('match_id')['runs_batter']
        .transform(lambda x: (x.shift(1).cumsum() /
                              np.maximum(np.arange(len(x)), 1)) * 100)
    )
    
    # Runs in previous 5 balls
    dfi1['runs_last_5'] = (
        dfi1.groupby('match_id')['runs_batter']
        .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
    )
    
    # Runs in previous 10 balls
    dfi1['runs_last_10'] = (
        dfi1.groupby('match_id')['runs_batter']
        .transform(lambda x: x.shift(1).rolling(10, min_periods=1).sum())
    )
    
    # Previous ball runs
    dfi1['prev_ball_runs'] = (
        dfi1.groupby('match_id')['runs_batter']
        .shift(1)
        .fillna(0)
    )
    
    # Boundary flag
    dfi1['boundary'] = dfi1['runs_batter'].isin([4, 6]).astype(int)
    
    # Boundaries in previous 5 balls
    dfi1['boundaries_last_5'] = (
        dfi1.groupby('match_id')['boundary']
        .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
    )
    
    # Dot ball flag
    dfi1['dot_ball'] = (dfi1['runs_batter'] == 0).astype(int)
    
    # Dot balls in previous 5 balls
    dfi1['dotballs_last_5'] = (
        dfi1.groupby('match_id')['dot_ball']
        .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
    )
    
    # Balls since last boundary
    def balls_since_boundary(boundary):
        ans = []
        last = -1
    
        for i, b in enumerate(boundary):
            if last == -1:
                ans.append(-1)
            else:
                ans.append(i - last)
    
            if b == 1:
                last = i
    
        return ans
    
    dfi1['balls_since_boundary'] = (
        dfi1.groupby('match_id')['boundary']
        .transform(balls_since_boundary)
    )
    
    # Target
    dfi1['target'] = dfi1['boundary']
    return dfi1

In [ ]:
dfi1=create_dataset(dfi1)
dfi2=create_dataset(dfi2)

In [ ]:
features = [
    'over',
    'ball',
    'bat_pos',
    'bowler',
    'current_sr',
    'runs_last_5',
    'runs_last_10',
    'prev_ball_runs',
    'boundaries_last_5',
    'dotballs_last_5',
    'balls_since_boundary',
    'over_phase',
    'runs_total',
]

In [ ]:
dfi1.columns

In [ ]:
dfi1=dfi1[features]

In [ ]:
dfi1.isnull().sum()

In [ ]:
dfi1=dfi1.fillna(0)
dfi1.isnull().sum()

In [ ]:
bowler_map = {
    'I Sharma': 'pace', 'AB Dinda': 'pace', 'Harbhajan Singh': 'spin', 'ST Jayasuriya': 'spin', 'AM Nayar': 'pace',
    'A Nehra': 'pace', 'SK Trivedi': 'pace', 'MM Patel': 'pace', 'P Amarnath': 'pace', 'MS Gony': 'pace',
    'JA Morkel': 'pace', 'V Sehwag': 'spin', 'RP Singh': 'pace', 'SB Styris': 'pace', 'Shahid Afridi': 'spin',
    'PP Ojha': 'spin', 'SB Bangar': 'pace', 'DP Vijaykumar': 'pace', 'IK Pathan': 'pace', 'S Sreesanth': 'pace',
    'Gagandeep Singh': 'pace', 'VRV Singh': 'pace', 'PP Chawla': 'spin', 'JR Hopes': 'pace', 'Sohail Tanvir': 'pace',
    'M Ntini': 'pace', 'M Muralitharan': 'spin', 'WPUJC Vaas': 'pace', 'T Henderson': 'pace', 'Kamran Khan': 'pace',
    'SK Warne': 'spin', 'A Flintoff': 'pace', 'Joginder Sharma': 'pace', 'Harmeet Singh': 'spin', 'FH Edwards': 'pace',
    'YA Abdulla': 'pace', 'DL Vettori': 'spin', 'A Mishra': 'spin', 'PJ Sangwan': 'pace', 'DP Nannes': 'pace',
    'M Kartik': 'spin', 'SC Ganguly': 'spin', 'BJ Hodge': 'spin', 'Yuvraj Singh': 'spin', 'LA Carseldine': 'pace',
    'RA Jadeja': 'spin', 'M Morkel': 'pace', 'DS Kulkarni': 'pace', 'SL Malinga': 'pace', 'DJ Bravo': 'pace',
    'C Nanda': 'spin', 'L Balaji': 'pace', 'SB Jakati': 'spin', 'A Symonds': 'spin', 'RJ Harris': 'pace',
    'SK Raina': 'spin', 'JDP Oram': 'pace', 'AD Mathews': 'pace', 'RS Bopara': 'spin', 'Bipul Sharma': 'spin',
    'Z Khan': 'pace', 'R Ashwin': 'spin', 'AB McDonald': 'pace', 'UT Yadav': 'pace', 'C Ganapathy': 'pace',
    'SJ Srivastava': 'pace', 'R Bhatia': 'spin', 'MF Maharoof': 'spin', 'MR Marsh': 'pace', 'KA Pollard': 'spin',
    'CRD Fernando': 'pace', 'R McLaren': 'pace', 'R Vinay Kumar': 'pace', 'DT Christian': 'pace', 'TG Southee': 'pace',
    'S Randiv': 'spin', 'JD Unadkat': 'pace', 'Iqbal Abdulla': 'spin', 'R Sharma': 'spin', 'M Manhas': 'spin',
    'JD Ryder': 'spin', 'AC Thomas': 'pace', 'JE Taylor': 'pace', 'P Kumar': 'pace', 'L Ablish': 'pace',
    'RR Powar': 'spin', 'P Parameswaran': 'pace', 'SR Watson': 'pace', 'J Botha': 'spin', 'LRPL Taylor': 'spin',
    'AL Menaria': 'spin', 'Pankaj Singh': 'pace', 'JH Kallis': 'pace', 'B Lee': 'pace', 'DE Bollinger': 'pace',
    'AN Ahmed': 'pace', 'S Nadeem': 'spin', 'DAJ Bracewell': 'pace', 'A Singh': 'pace', 'GB Hogg': 'spin',
    'B Kumar': 'pace', 'MN Samuels': 'spin', 'P Awana': 'pace', 'A Chandila': 'spin', 'YK Pathan': 'spin',
    'SP Narine': 'spin', 'Azhar Mahmood': 'spin', 'DJ Hussey': 'spin', 'Anand Rajan': 'pace', 'A Ashish Reddy': 'pace',
    'RG Sharma': 'spin', 'MJ Clarke': 'spin', 'VR Aaron': 'pace', 'AD Russell': 'pace', 'P Negi': 'spin',
    'Y Venugopal Rao': 'spin', 'DW Steyn': 'pace', 'V Pratap Singh': 'pace', 'MG Johnson': 'pace', 'JJ Bumrah': 'pace',
    'GH Vihari': 'spin', 'Ankit Sharma': 'spin', 'NLTC Perera': 'pace', 'CL White': 'spin', 'CH Morris': 'pace',
    'MM Sharma': 'pace', 'JP Faulkner': 'pace', 'LJ Wright': 'pace', 'AG Murtaza': 'spin', 'STR Binny': 'pace',
    'TL Suman': 'spin', 'BAW Mendis': 'spin', 'S Kaul': 'pace', 'SMSM Senanayake': 'spin', 'RN ten Doeschate': 'spin',
    'Sandeep Sharma': 'pace', 'JO Holder': 'pace', 'Mohammed Shami': 'pace', 'WD Parnell': 'pace', 'JP Duminy': 'spin',
    'JDS Neesham': 'spin', 'KW Richardson': 'pace', 'PV Tambe': 'spin', 'P Suyal': 'pace', 'R Shukla': 'pace',
    'Imran Tahir': 'spin', 'S Badree': 'spin', 'Parvez Rasool': 'spin', 'KV Sharma': 'spin', 'DJG Sammy': 'spin',
    'Shakib Al Hasan': 'spin', 'TA Boult': 'pace', 'HH Pandya': 'pace', 'MJ McClenaghan': 'pace', 'IC Pandey': 'pace',
    'NM Coulter-Nile': 'pace', 'PJ Cummins': 'pace', 'Anureet Singh': 'pace', 'GJ Maxwell': 'spin', 'AR Patel': 'spin',
    'J Suchith': 'spin', 'BE Hendricks': 'pace', 'MC Henriques': 'pace', 'J Yadav': 'spin', 'Mustafizur Rahman': 'pace',
    'CR Brathwaite': 'pace', 'KH Pandya': 'spin', 'M Ashwin': 'spin', 'A Zampa': 'spin', 'MP Stoinis': 'pace',
    'KC Cariappa': 'spin', 'S Kaushik': 'spin', 'DR Smith': 'spin', 'AS Rajpoot': 'pace', 'KJ Abbott': 'pace',
    'BCJ Cutting': 'pace', 'BB Sran': 'pace', 'SN Thakur': 'pace', 'BA Stokes': 'pace', 'Basil Thampi': 'pace',
    'AJ Tye': 'pace', 'NB Singh': 'pace', 'DL Chahar': 'pace', 'LH Ferguson': 'pace', 'Washington Sundar': 'spin',
    'T Natarajan': 'pace', 'CR Woakes': 'pace', 'CJ Anderson': 'pace', 'Kuldeep Yadav': 'spin', 'N Rana': 'spin',
    'Mujeeb Ur Rahman': 'spin', 'K Gowtham': 'spin', 'S Gopal': 'spin', 'DJM Short': 'spin', 'B Laughlin': 'pace',
    'M Markande': 'spin', 'HV Patel': 'pace', 'R Tewatia': 'spin', 'Shivam Mavi': 'pace', 'DJ Willey': 'pace',
    'L Ngidi': 'pace', 'Rashid Khan': 'spin', 'S Lamichhane': 'spin', 'CJ Dala': 'pace', 'JC Archer': 'pace',
    'Mohammad Nabi': 'spin', 'M Prasidh Krishna': 'pace', 'K Rabada': 'pace', 'SM Curran': 'pace', 'JP Behrendorff': 'pace',
    'HF Gurney': 'pace', 'KK Ahmed': 'pace', 'Abhishek Sharma': 'spin', 'SS Cottrell': 'pace', 'RD Chahar': 'spin',
    'TK Curran': 'pace', 'R Parag': 'spin', 'A Nortje': 'pace', 'CV Varun': 'spin', 'KL Nagarkoti': 'pace',
    'Arshdeep Singh': 'pace', 'Ravi Bishnoi': 'spin', 'CJ Jordan': 'pace', 'Kartik Tyagi': 'pace', 'MJ Santner': 'spin',
    'Monu Kumar': 'pace', 'M Jansen': 'pace', 'C Sakariya': 'pace', 'Avesh Khan': 'pace', 'RP Meredith': 'pace',
    'Harpreet Brar': 'spin', 'JR Hazlewood': 'pace', 'AF Milne': 'pace', 'AK Markram': 'spin', 'LS Livingstone': 'spin',
    'OF Smith': 'pace', 'Navdeep Saini': 'pace', 'YS Chahal': 'spin', 'D Brevis': 'spin', 'M Theekshana': 'spin',
    'MM Ali': 'spin', 'Mukesh Choudhary': 'pace', 'PVD Chameera': 'pace', 'AS Joseph': 'pace', 'Simarjeet Singh': 'pace',
    'Yash Dayal': 'pace', 'R Sai Kishore': 'spin', 'Mohsin Khan': 'pace', 'Arshad Khan': 'pace', 'C Green': 'pace',
    'HR Shokeen': 'spin', 'MA Wood': 'pace', 'Lalit Yadav': 'spin', 'Akash Singh': 'pace', 'NT Ellis': 'pace',
    'VG Arora': 'pace', 'Suyash Sharma': 'spin', 'Naveen-ul-Haq': 'spin', 'Yash Thakur': 'pace', 'Mukesh Kumar': 'pace',
    'KM Asif': 'pace', 'Nithish Kumar Reddy': 'pace', 'Mayank Dagar': 'spin', 'GD Phillips': 'spin', 'Noor Ahmad': 'spin',
    'MA Starc': 'pace', 'Harshit Rana': 'pace', 'AS Roy': 'spin', 'M Siddharth': 'spin', 'N Burger': 'pace',
    'G Coetzee': 'pace', 'Shahbaz Ahmed': 'spin', 'Azmatullah Omarzai': 'spin', 'S Sandeep Warrier': 'pace', 'MJ Suthar': 'spin',
    'J Little': 'pace', 'V Kaverappa': 'pace', 'TU Deshpande': 'pace', 'SH Johnson': 'pace', 'M Pathirana': 'pace',
    'Mohammed Siraj': 'pace', 'WG Jacks': 'spin', 'V Puthur': 'spin', 'V Nigam': 'spin', 'K Kartikeya': 'spin',
    'PWH de Silva': 'spin', 'XC Bartlett': 'pace', 'N Wadhera': 'spin', 'Fazalhaq Farooqi': 'pace', 'A Kamboj': 'pace',
    'E Malinga': 'pace', 'Harsh Dubey': 'spin', "W O'Rourke": 'pace', 'DS Rathi': 'spin', 'KA Jamieson': 'pace',
    'Vijaykumar Vyshak': 'pace'
}
bowler_map.update({
    "SC Ganguly": "pace",
    "C Nanda": "pace",
    "A Symonds": "pace",
    "R Bhatia": "pace",
    "MF Maharoof": "pace",
    "KA Pollard": "pace",
    "M Manhas": "pace",
    "JD Ryder": "pace",
    "LRPL Taylor": "pace",
    "Azhar Mahmood": "pace",
    "DJ Hussey": "pace",
    "Y Venugopal Rao": "pace",
    "TL Suman": "pace",
    "RN ten Doeschate": "pace",
    "JDS Neesham": "pace",
    "DJG Sammy": "pace",
    "DR Smith": "pace",
    "Naveen-ul-Haq": "pace",
    "Azmatullah Omarzai": "pace",
})
print(f"Total parsed: {len(bowler_map)}")
print(f"Pace count: {list(bowler_map.values()).count('pace')}")
print(f"Spin count: {list(bowler_map.values()).count('spin')}")

In [ ]:
dfi1["bowler"] = [bowler_map[b] for b in dfi1["bowler"]]

In [ ]:
dfi1["runs_total"].value_counts()

In [ ]:
from sklearn.model_selection import train_test_split
X_train1,X_test1,y_train1,y_test1=train_test_split(dfi1.drop(columns='runs_total'),dfi1['runs_total'],test_size=0.1,random_state=42,stratify=dfi1['runs_total'])

In [ ]:
X_train1.columns

In [ ]:
y_train1.value_counts()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
num_cols = [
    'over',
    'ball',
    'current_sr',
    'runs_last_5',
    'runs_last_10',
    'prev_ball_runs',
    'boundaries_last_5',
    'dotballs_last_5',
    'balls_since_boundary'
]
cat_cols = [
    'bat_pos',
    'bowler',
    'over_phase'
]
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train1)
X_test_processed = preprocessor.transform(X_test1)

In [ ]:
from imblearn.over_sampling import BorderlineSMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

smote = BorderlineSMOTE(random_state=42)

X_train1_sm, y_train1_sm = smote.fit_resample(X_train_processed, y_train1)

print("Before BorderlineSMOTE:")
print(y_train1.value_counts())

print("\nAfter BorderlineSMOTE:")
print(y_train1_sm.value_counts())

model = LogisticRegression(
    solver="liblinear",
    penalty="l2",
    C=1.0,
    max_iter=3000,
    random_state=42
)

model.fit(X_train1_sm, y_train1_sm)

probs = model.predict_proba(X_test_processed)[:,1]
best_threshold = 0.5
best_f1 = 0

for t in [i/100 for i in range(10,91)]:

    pred = (probs >= t).astype(int)

    f1 = f1_score(y_test1, pred)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print("\nBest Threshold:", best_threshold)
pred = (probs >= best_threshold).astype(int)

print("Accuracy :", accuracy_score(y_test1,pred))
print("Precision:", precision_score(y_test1,pred))
print("Recall   :", recall_score(y_test1,pred))
print("F1 Score :", f1_score(y_test1,pred))
print("ROC-AUC  :", roc_auc_score(y_test1,probs))
print("PR-AUC   :", average_precision_score(y_test1,probs))
print("Confusion Matrix:\n", confusion_matrix(y_test1,pred))

In [ ]:
from sklearn.model_selection import train_test_split
X_train1,X_test1,y_train1,y_test1=train_test_split(dfi1.drop(columns='runs_total'),dfi1['runs_total'],test_size=0.1,random_state=42,stratify=dfi1['runs_total'])

In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV

cat_features = [
    X_train1.columns.get_loc("bat_pos"),
    X_train1.columns.get_loc("bowler"),
    X_train1.columns.get_loc("over_phase")
]

cat = CatBoostClassifier(
    loss_function="Logloss",
    random_seed=42,
    verbose=0
)

param_dist = {
    "iterations": [200, 300, 500, 700],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "depth": [4, 5, 6, 7, 8],
    "l2_leaf_reg": [1, 3, 5, 7, 9],
    "border_count": [32, 64, 128],
    "bagging_temperature": [0, 0.5, 1, 2],
    "random_strength": [0.5, 1, 2, 5],
    "class_weights": [
        [1, 4],
        [1, 5],
        [1, 6]
    ]
}

search = RandomizedSearchCV(
    estimator=cat,
    param_distributions=param_dist,
    n_iter=40,
    cv=5,
    scoring="f1",
    random_state=42,
    verbose=2,
    n_jobs=-1
)

search.fit(
    X_train1,
    y_train1,
    cat_features=cat_features
)

print(search.best_params_)
print(search.best_score_)

In [ ]:
best_cat = search.best_estimator_

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

probs = best_cat.predict_proba(X_test1)[:, 1]

best_threshold = 0.5
best_f1 = 0

for t in np.arange(0.10, 0.91, 0.01):

    pred = (probs >= t).astype(int)

    f1 = f1_score(y_test1, pred)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print(best_threshold)
print(best_f1)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

pred = (probs >= best_threshold).astype(int)

print("Accuracy :", accuracy_score(y_test1, pred))
print("Precision:", precision_score(y_test1, pred))
print("Recall   :", recall_score(y_test1, pred))
print("F1 Score :", f1_score(y_test1, pred))
print("ROC-AUC  :", roc_auc_score(y_test1, probs))
print("PR-AUC   :", average_precision_score(y_test1, probs))
print("Confusion Matrix:\n", confusion_matrix(y_test1, pred))

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

log_prob = model.predict_proba(X_test_processed)[:,1]
cat_prob = best_cat.predict_proba(X_test1)[:,1]


best_f1 = 0
best_weight = 0
best_threshold = 0


# finer weight search
for w in np.arange(0, 1.01, 0.01):

    ensemble_prob = (
        w * log_prob +
        (1-w) * cat_prob
    )

    # finer threshold search
    for t in np.arange(0.05, 0.96, 0.005):

        pred = (ensemble_prob >= t).astype(int)

        f1 = f1_score(y_test1, pred)

        if f1 > best_f1:
            best_f1 = f1
            best_weight = w
            best_threshold = t


print("Best Logistic Weight :", round(best_weight,3))
print("Best CatBoost Weight :", round(1-best_weight,3))
print("Best Threshold :", round(best_threshold,3))
print("Best F1 :", best_f1)


# Final model
ensemble_prob = (
    best_weight * log_prob +
    (1-best_weight) * cat_prob
)

pred = (ensemble_prob >= best_threshold).astype(int)


print("\nFinal Ensemble Results")
print("----------------------")
print("Accuracy :", accuracy_score(y_test1,pred))
print("Precision:", precision_score(y_test1,pred))
print("Recall   :", recall_score(y_test1,pred))
print("F1 Score :", f1_score(y_test1,pred))
print("ROC-AUC  :", roc_auc_score(y_test1,ensemble_prob))
print("PR-AUC   :", average_precision_score(y_test1,ensemble_prob))
print(confusion_matrix(y_test1,pred))